In [ ]:
# ==============================================================================
# STEP 1: INSTALL NECESSARY LIBRARIES
# ==============================================================================
# We use -q to make the installation logs less noisy.
!pip install -q transformers[torch] datasets sentencepiece accelerate

# ==============================================================================
# STEP 2: IMPORT LIBRARIES
# ==============================================================================
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    pipeline
)

# ==============================================================================
# STEP 3: CONFIGURE MODEL AND DATASET PARAMETERS
# ==============================================================================
# We'll fine-tune Google's mT5-small model. It's multilingual and a good starting point.
model_checkpoint = "google/mt5-small"

# The dataset is ai4bharat/samanantar, which has many Indic language pairs.
# We specify the Hindi ('hi') to Tamil ('ta') pair.
source_lang = "hi"
target_lang = "ta"
dataset_pair = f"{source_lang}-{target_lang}"

# ==============================================================================
# STEP 4: LOAD THE DATASET
# ==============================================================================
print("🚀 Loading dataset...")
# IMPORTANT: The full dataset is very large. We'll use just 1% for this demo.
# For a real, high-quality model, increase this percentage (e.g., 'train[:20%]')
# or remove the slice completely to use the full dataset.
raw_datasets = load_dataset("ai4bharat/samanantar", dataset_pair, split='train[:1%]')
print("✅ Dataset loaded.")
print("\nHere's a sample from the dataset:")
print(raw_datasets[0])

# ==============================================================================
# STEP 5: PREPROCESS THE DATA (TOKENIZATION)
# ==============================================================================
print("\n🚀 Initializing tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
print("✅ Tokenizer initialized.")

# The prompt mT5 uses to know what to do. This prefix is crucial.
prefix = f"translate {source_lang} to {target_lang}: "

def preprocess_function(examples):
    """Tokenizes the source and target texts."""
    # Prepare the inputs by adding the prefix
    inputs = [prefix + ex[source_lang] for ex in examples["translation"]]
    # Prepare the target labels
    targets = [ex[target_lang] for ex in examples["translation"]]

    # Tokenize both
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(text_target=targets, max_length=128, truncation=True, padding="max_length")

    # Set the labels for the model
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("\n🚀 Applying preprocessing to the dataset...")
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)
# Remove the original 'translation' column as it's no longer needed
tokenized_datasets = tokenized_datasets.remove_columns(["translation"])
print("✅ Preprocessing complete.")

# ==============================================================================
# STEP 6: FINE-TUNE THE MODEL
# ==============================================================================
print("\n🚀 Loading pre-trained model...")
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
print("✅ Model loaded.")

# Define training arguments
batch_size = 16
model_name = "mt5-hindi-to-tamil-translator"

args = Seq2SeqTrainingArguments(
    output_dir=model_name,
    evaluation_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=1,  # Increase to 3-5 for a better model
    predict_with_generate=True,
    fp16=True,  # Use mixed-precision for speedup on GPU
)

# The data collator prepares batches of data for training
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# The Trainer handles the entire training loop
trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

# Start training!
print("\n🚀 Starting the fine-tuning process...")
trainer.train()
print("✅ Training complete!")

# Save the final model for later use
final_model_path = f"./{model_name}-final"
trainer.save_model(final_model_path)
print(f"✅ Model saved to {final_model_path}")


# ==============================================================================
# STEP 7: USE THE TRAINED MODEL FOR TRANSLATION (INFERENCE)
# ==============================================================================
print("\n\n--- INFERENCE ---")
print("🚀 Loading fine-tuned model for translation...")

# Load the model we just trained using the pipeline function
translator = pipeline("translation", model=final_model_path, tokenizer=tokenizer)
print("✅ Translator pipeline ready.")

# Example sentences in Hindi to translate
hindi_sentences = [
    "आपका नाम क्या है?",           # What is your name?
    "भारत एक खूबसूरत देश है।",       # India is a beautiful country.
    "मैं कल स्कूल जाऊँगा।",          # I will go to school tomorrow.
    "यह किताब बहुत दिलचस्प है।"    # This book is very interesting.
]

print("\n--- TRANSLATION RESULTS ---")
for sentence in hindi_sentences:
    # Remember to add the prefix!
    prompt = f"translate {source_lang} to {target_lang}: {sentence}"
    result = translator(prompt)

    print(f"\nHindi: {sentence}")
    print(f"Tamil: {result[0]['translation_text']}")

print("\n--- END OF SCRIPT ---")

ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_

ValueError: BuilderConfig 'hi-ta' not found. Available: ['as', 'bn', 'gu', 'hi', 'kn', 'ml', 'mr', 'or', 'pa', 'ta', 'te']

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

#
# SPDX-License-Identifier: GPL-3.0
#
# GNU Radio Python Flow Graph
# Title: Not titled yet
# GNU Radio version: 3.10.12.0

from PyQt5 import Qt
from gnuradio import qtgui
from PyQt5 import QtCore
from PyQt5.QtCore import QObject, pyqtSlot
from gnuradio import analog
from gnuradio import audio
from gnuradio import blocks
from gnuradio import filter
from gnuradio.filter import firdes
from gnuradio import gr
from gnuradio.fft import window
import sys
import signal
from PyQt5 import Qt
from argparse import ArgumentParser
from gnuradio.eng_arg import eng_float, intx
from gnuradio import eng_notation
import sip
import threading



class test(gr.top_block, Qt.QWidget):

    def __init__(self):
        gr.top_block.__init__(self, "Not titled yet", catch_exceptions=True)
        Qt.QWidget.__init__(self)
        self.setWindowTitle("Not titled yet")
        qtgui.util.check_set_qss()
        try:
            self.setWindowIcon(Qt.QIcon.fromTheme('gnuradio-grc'))
        except BaseException as exc:
            print(f"Qt GUI: Could not set Icon: {str(exc)}", file=sys.stderr)
        self.top_scroll_layout = Qt.QVBoxLayout()
        self.setLayout(self.top_scroll_layout)
        self.top_scroll = Qt.QScrollArea()
        self.top_scroll.setFrameStyle(Qt.QFrame.NoFrame)
        self.top_scroll_layout.addWidget(self.top_scroll)
        self.top_scroll.setWidgetResizable(True)
        self.top_widget = Qt.QWidget()
        self.top_scroll.setWidget(self.top_widget)
        self.top_layout = Qt.QVBoxLayout(self.top_widget)
        self.top_grid_layout = Qt.QGridLayout()
        self.top_layout.addLayout(self.top_grid_layout)

        self.settings = Qt.QSettings("gnuradio/flowgraphs", "test")

        try:
            geometry = self.settings.value("geometry")
            if geometry:
                self.restoreGeometry(geometry)
        except BaseException as exc:
            print(f"Qt GUI: Could not restore geometry: {str(exc)}", file=sys.stderr)
        self.flowgraph_started = threading.Event()

        ##################################################
        # Variables
        ##################################################
        self.volume = volume = 5
        self.samp_rate = samp_rate = 48000
        self.pl_freq = pl_freq = 0
        self.interp = interp = 10
        self.ifrate = ifrate = 192000
        self.URSP_RATE = URSP_RATE = 576000

        ##################################################
        # Blocks
        ##################################################

        self._volume_range = qtgui.Range(0, 10, 0.1, 5, 200)
        self._volume_win = qtgui.RangeWidget(self._volume_range, self.set_volume, "Audio gain", "counter_slider", float, QtCore.Qt.Horizontal)
        self.top_layout.addWidget(self._volume_win)
        self.qtgui_sink_x_0 = qtgui.sink_c(
            1024, #fftsize
            window.WIN_BLACKMAN_hARRIS, #wintype
            0, #fc
            samp_rate, #bw
            "", #name
            True, #plotfreq
            True, #plotwaterfall
            True, #plottime
            True, #plotconst
            None # parent
        )
        self.qtgui_sink_x_0.set_update_time(1.0/10)
        self._qtgui_sink_x_0_win = sip.wrapinstance(self.qtgui_sink_x_0.qwidget(), Qt.QWidget)

        self.qtgui_sink_x_0.enable_rf_freq(False)

        self.top_layout.addWidget(self._qtgui_sink_x_0_win)
        # Create the options list
        self._pl_freq_options = [0.0, 67.0, 71.9, 74.4, 77.0, 79.7, 82.5, 85.4, 88.5, 91.5, 94.8, 97.4, 100.0, 103.5, 107.2, 110.9, 114.8, 118.8, 123.0, 127.3, 131.8, 136.5, 141.3, 146.2, 151.4, 156.7, 162.2, 167.9, 173.8, 179.9, 186.2, 192.8, 203.5, 210.7, 218.1, 225.7, 233.6, 241.8, 250.3]
        # Create the labels list
        self._pl_freq_labels = ['0.0', '67.0', '71.9', '74.4', '77.0', '79.7', '82.5', '85.4', '88.5', '91.5', '94.8', '97.4', '100.0', '103.5', '107.2', '110.9', '114.8', '118.8', '123.0', '127.3', '131.8', '136.5', '141.3', '146.2', '151.4', '156.7', '162.2', '167.9', '173.8', '179.9', '186.2', '192.8', '203.5', '210.7', '218.1', '225.7', '233.6', '241.8', '250.3']
        # Create the combo box
        self._pl_freq_tool_bar = Qt.QToolBar(self)
        self._pl_freq_tool_bar.addWidget(Qt.QLabel("PL TONE" + ": "))
        self._pl_freq_combo_box = Qt.QComboBox()
        self._pl_freq_tool_bar.addWidget(self._pl_freq_combo_box)
        for _label in self._pl_freq_labels: self._pl_freq_combo_box.addItem(_label)
        self._pl_freq_callback = lambda i: Qt.QMetaObject.invokeMethod(self._pl_freq_combo_box, "setCurrentIndex", Qt.Q_ARG("int", self._pl_freq_options.index(i)))
        self._pl_freq_callback(self.pl_freq)
        self._pl_freq_combo_box.currentIndexChanged.connect(
            lambda i: self.set_pl_freq(self._pl_freq_options[i]))
        # Create the radio buttons
        self.top_layout.addWidget(self._pl_freq_tool_bar)
        self.low_pass_filter_0 = filter.fir_filter_ccf(
            1,
            firdes.low_pass(
                1,
                ifrate,
                5000,
                2000,
                window.WIN_HAMMING,
                6.76))
        self.blocks_multiply_const_vxx_0 = blocks.multiply_const_ff(5)
        self.blocks_add_xx_0 = blocks.add_vff(1)
        self.band_pass_filter_0 = filter.interp_fir_filter_fff(
            1,
            firdes.band_pass(
                1,
                48000,
                300,
                5000,
                200,
                window.WIN_HAMMING,
                6.76))
        self.audio_source_0 = audio.source(48000, '', True)
        self.analog_sig_source_x_0 = analog.sig_source_f(samp_rate, analog.GR_SIN_WAVE, 0, 0.15, 0, 0)
        self.analog_nbfm_tx_0 = analog.nbfm_tx(
         audio_rate=samp_rate,
         quad_rate=ifrate,
         tau=(75e-6),
         max_dev=5000,
         fh=(-1.0),
                )


        ##################################################
        # Connections
        ##################################################
        self.connect((self.analog_nbfm_tx_0, 0), (self.low_pass_filter_0, 0))
        self.connect((self.analog_sig_source_x_0, 0), (self.blocks_add_xx_0, 1))
        self.connect((self.audio_source_0, 0), (self.band_pass_filter_0, 0))
        self.connect((self.band_pass_filter_0, 0), (self.blocks_multiply_const_vxx_0, 0))
        self.connect((self.blocks_add_xx_0, 0), (self.analog_nbfm_tx_0, 0))
        self.connect((self.blocks_multiply_const_vxx_0, 0), (self.blocks_add_xx_0, 0))
        self.connect((self.low_pass_filter_0, 0), (self.qtgui_sink_x_0, 0))


    def closeEvent(self, event):
        self.settings = Qt.QSettings("gnuradio/flowgraphs", "test")
        self.settings.setValue("geometry", self.saveGeometry())
        self.stop()
        self.wait()

        event.accept()

    def get_volume(self):
        return self.volume

    def set_volume(self, volume):
        self.volume = volume

    def get_samp_rate(self):
        return self.samp_rate

    def set_samp_rate(self, samp_rate):
        self.samp_rate = samp_rate
        self.analog_sig_source_x_0.set_sampling_freq(self.samp_rate)
        self.qtgui_sink_x_0.set_frequency_range(0, self.samp_rate)

    def get_pl_freq(self):
        return self.pl_freq

    def set_pl_freq(self, pl_freq):
        self.pl_freq = pl_freq
        self._pl_freq_callback(self.pl_freq)

    def get_interp(self):
        return self.interp

    def set_interp(self, interp):
        self.interp = interp

    def get_ifrate(self):
        return self.ifrate

    def set_ifrate(self, ifrate):
        self.ifrate = ifrate
        self.low_pass_filter_0.set_taps(firdes.low_pass(1, self.ifrate, 5000, 2000, window.WIN_HAMMING, 6.76))

    def get_URSP_RATE(self):
        return self.URSP_RATE

    def set_URSP_RATE(self, URSP_RATE):
        self.URSP_RATE = URSP_RATE




def main(top_block_cls=test, options=None):

    qapp = Qt.QApplication(sys.argv)

    tb = top_block_cls()

    tb.start()
    tb.flowgraph_started.set()

    tb.show()

    def sig_handler(sig=None, frame=None):
        tb.stop()
        tb.wait()

        Qt.QApplication.quit()

    signal.signal(signal.SIGINT, sig_handler)
    signal.signal(signal.SIGTERM, sig_handler)

    timer = Qt.QTimer()
    timer.start(500)
    timer.timeout.connect(lambda: None)

    qapp.exec_()

if __name__ == '__main__':
    main()

ModuleNotFoundError: No module named 'PyQt5'

In [ ]:
# ==============================================================================
# STEP 1: INSTALL NECESSARY LIBRARIES
# ==============================================================================
# We use -q to make the installation logs less noisy.
!pip install -q transformers[torch] datasets sentencepiece accelerate

# ==============================================================================
# STEP 2: IMPORT LIBRARIES
# ==============================================================================
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    pipeline
)

# ==============================================================================
# STEP 3: CONFIGURE MODEL AND DATASET PARAMETERS
# ==============================================================================
# We'll fine-tune Google's mT5-small model. It's multilingual and a good starting point.
model_checkpoint = "google/mt5-small"

# Define our source and target languages
source_lang = "hi"
target_lang = "ta"
intermediate_lang = "en" # We'll use English as an intermediate language

# ==============================================================================
# STEP 4: LOAD THE DATASET (USING INTERMEDIATE LANGUAGE)
# ==============================================================================
print("🚀 Loading datasets...")
# We need to load both en-hi and en-ta pairs and join them
dataset_pair_hi_en = f"{intermediate_lang}-{source_lang}"
dataset_pair_en_ta = f"{intermediate_lang}-{target_lang}"


# IMPORTANT: The full dataset is very large. We'll use just 1% for this demo.
# For a real, high-quality model, increase this percentage (e.g., 'train[:20%]')
# or remove the slice completely to use the full dataset.
raw_datasets_hi_en = load_dataset("opus100", dataset_pair_hi_en, split='train[:1%]')
raw_datasets_en_ta = load_dataset("opus100", dataset_pair_en_ta, split='train[:1%]')

print("✅ Datasets loaded.")

# Now, we need to join these datasets based on the English translation.
# We'll convert them to pandas DataFrames for easier merging.
import pandas as pd

print("\n🚀 Merging datasets...")

# Convert to pandas DataFrames
df_hi_en = raw_datasets_hi_en.to_pandas()
df_en_ta = raw_datasets_en_ta.to_pandas()

# Flatten the 'translation' column
df_hi_en['en'] = df_hi_en['translation'].apply(lambda x: x['en'])
df_hi_en['hi'] = df_hi_en['translation'].apply(lambda x: x['hi'])
df_en_ta['en'] = df_en_ta['translation'].apply(lambda x: x['en'])
df_en_ta['ta'] = df_en_ta['translation'].apply(lambda x: x['ta'])

# Merge on the English column
# We use an inner merge to keep only pairs that exist in both datasets
merged_df = pd.merge(df_hi_en[['en', 'hi']], df_en_ta[['en', 'ta']], on='en', how='inner')

# Convert back to a Dataset object
from datasets import Dataset
raw_datasets = Dataset.from_pandas(merged_df)

# We can remove the intermediate English column if not needed for training directly
# raw_datasets = raw_datasets.remove_columns(["en"])


print("✅ Datasets merged.")
print("\nHere's a sample from the merged dataset:")
print(raw_datasets[0])


# ==============================================================================
# STEP 5: PREPROCESS THE DATA (TOKENIZATION)
# ==============================================================================
print("\n🚀 Initializing tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
print("✅ Tokenizer initialized.")

# The prompt mT5 uses to know what to do. This prefix is crucial.
prefix = f"translate {source_lang} to {target_lang}: "

def preprocess_function(examples):
    """Tokenizes the source and target texts from the merged dataset."""
    # The merged dataset now has 'hi' and 'ta' columns directly
    inputs = [prefix + examples[source_lang][i] for i in range(len(examples[source_lang]))]
    targets = [examples[target_lang][i] for i in range(len(examples[target_lang]))]

    # Tokenize both
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(text_target=targets, max_length=128, truncation=True, padding="max_length")

    # Set the labels for the model
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("\n🚀 Applying preprocessing to the dataset...")
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)
# The original 'hi' and 'ta' columns are now implicitly handled by the tokenization
# No need to explicitly remove them unless you want a cleaner dataset object
print("✅ Preprocessing complete.")

# ==============================================================================
# STEP 6: FINE-TUNE THE MODEL
# ==============================================================================
print("\n🚀 Loading pre-trained model...")
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
print("✅ Model loaded.")

# Define training arguments
batch_size = 4 # Further reduced batch size
model_name = "mt5-hindi-to-tamil-translator"

args = Seq2SeqTrainingArguments(
    output_dir=model_name,
    # evaluation_strategy="no", # Removed as it's not a valid argument
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=1,  # Increase to 3-5 for a better model
    predict_with_generate=True,
    fp16=True,  # Use mixed-precision for speedup on GPU
    report_to="none" # Disable wandb logging
    # Push to Hub is useful if you want to save the model to your Hugging Face account
    # push_to_hub=True,
    # hub_model_id=f"your_username/{model_name}",
    # hub_token=userdata.get("HF_TOKEN"), # Assuming you have set your HF token in Colab secrets
)

# The data collator prepares batches of data for training
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# The Trainer handles the entire training loop
trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

# Start training!
print("\n🚀 Starting the fine-tuning process...")
trainer.train()
print("✅ Training complete!")

# Save the final model for later use
final_model_path = f"./{model_name}-final"
trainer.save_model(final_model_path)
print(f"✅ Model saved to {final_model_path}")


# ==============================================================================
# STEP 7: USE THE TRAINED MODEL FOR TRANSLATION (INFERENCE)
# ==============================================================================
print("\n\n--- INFERENCE ---")
print("🚀 Loading fine-tuned model for translation...")

# Load the model we just trained using the pipeline function
translator = pipeline("translation", model=final_model_path, tokenizer=tokenizer)
print("✅ Translator pipeline ready.")

# Example sentences in Hindi to translate
hindi_sentences = [
    "आपका नाम क्या है?",           # What is your name?
    "भारत एक खूबसूरत देश है।",       # India is a beautiful country.
    "मैं कल स्कूल जाऊँगा।",          # I will go to school tomorrow.
    "यह किताब बहुत दिलचस्प है。"    # This book is very interesting.
]

print("\n--- TRANSLATION RESULTS ---")
for sentence in hindi_sentences:
    # Remember to add the prefix!
    prompt = f"translate {source_lang} to {target_lang}: {sentence}"
    result = translator(prompt)

    print(f"\nHindi: {sentence}")
    print(f"Tamil: {result[0]['translation_text']}")

print("\n--- END OF SCRIPT ---")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.9/193.9 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.4/224.4 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.3/377.3 kB 32.9 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/torch_xla/experimental/gru.py:113: SyntaxWarning: invalid escape sequence '\_'
  * **h_n**: tensor of shape :math:`(D * \text{num\_layers}, H_{out})` or
/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:82: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


🚀 Loading datasets...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Model loaded.


/tmp/ipython-input-3220859690.py:143: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


ValueError: fp16 mixed precision requires a GPU (not 'xla').

sheik syed test code

In [ ]:
# ============================
# STEP 1: Install dependencies
# ============================
!pip install transformers sentencepiece

# ============================
# STEP 2: Import libraries
# ============================
from transformers import MarianMTModel, MarianTokenizer

# ============================
# STEP 3: Choose a translation model
# Example: English to Tamil
# You can change to other language pairs from Hugging Face model hub
# ============================
model_name = "Helsinki-NLP/opus-mt-en-ta"  # English to Tamil
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# ============================
# STEP 4: Write a simple translation function
# ============================
def translate_text(text):
    # Tokenize the input text
    inputs = tokenizer([text], return_tensors="pt", padding=True)
    # Generate translated text
    translated = model.generate(**inputs)
    # Decode the output
    result = tokenizer.decode(translated[0], skip_special_tokens=True)
    return result

# ============================
# STEP 5: Test the translator
# ============================
text = "He kicked the bucket."  # Example sentence
translated_text = translate_text(text)
print("Original:", text)
print("Translated:", translated_text)

OSError: Helsinki-NLP/opus-mt-en-ta is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [ ]:
# Colab cell 1 — install
!pip install -q transformers sentencepiece

# Colab cell 2 — code
import torch
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

# Model to use (mBART-50 supports Tamil as ta_IN)
MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"

# Load tokenizer and model
tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
model = MBartForConditionalGeneration.from_pretrained(MODEL_NAME)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Quick check: available language codes (you should see 'en_XX' and 'ta_IN')
print("Some language codes (examples):", [k for k in tokenizer.lang_code_to_id.keys() if k in ("en_XX","ta_IN")])

def translate(text, src_lang="en_XX", tgt_lang="ta_IN", num_beams=4, max_length=128):
    """
    Translate `text` from src_lang to tgt_lang using mBART-50.
    src_lang/tgt_lang examples: "en_XX" (English), "ta_IN" (Tamil).
    """
    # tell tokenizer what the source language is
    tokenizer.src_lang = src_lang

    # tokenize and move tensors to the same device as the model
    encoded = tokenizer(text, return_tensors="pt", padding=True)
    encoded = {k: v.to(device) for k, v in encoded.items()}

    # force the model to generate in the target language by setting forced_bos_token_id
    forced_bos_token_id = tokenizer.lang_code_to_id[tgt_lang]

    generated_tokens = model.generate(
        **encoded,
        forced_bos_token_id=forced_bos_token_id,
        num_beams=num_beams,
        max_length=max_length,
        early_stopping=True,
        no_repeat_ngram_size=2
    )

    # decode and return the result
    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

# Examples
examples = [
    "He kicked the bucket.",
    "I went to the bank to withdraw some money.",
    "Can you help me book a doctor's appointment for tomorrow?"
]

for s in examples:
    out = translate(s, src_lang="en_XX", tgt_lang="ta_IN")
    print("EN:", s)
    print("TA:", out)
    print("-" * 50)

# Example: translate a paragraph (context-aware by giving more text)
paragraph = (
    "Ravi had been ill for a long time. His family cared for him daily. "
    "Last week, sadly, he kicked the bucket and the whole village mourned."
)
print("Paragraph -> Tamil:")
print(translate(paragraph, src_lang="en_XX", tgt_lang="ta_IN", num_beams=6, max_length=256))


language detection

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
data = pd.read_csv("language.csv")
x=np.array(data['Text'])
y=np.array(data['language'])
cv=CountVectorizer()
X=cv.fit_transform(x)
X_train,X_test, y_train,y_test = train_test_split(X,y, test_size= 0.33, random_state = 42)
model=MultinomialNB()
model.fit(X_train,y_train)
model.score(X_test,y_test)

FileNotFoundError: [Errno 2] No such file or directory: 'language.csv'

In [ ]:
user=input('Enter a text :')
data=cv.transform([user]).toarray()
output=model.predict(data)
print(output)

Enter a text :hi


NameError: name 'cv' is not defined

language translator

In [ ]:
# Import the GoogleTranslator class from the new library
!pip install deep_translator
from deep_translator import GoogleTranslator

# --- Step 1: Set your languages and text here ---
source_language = 'ta'      # 'en' for English
destination_language = 'hi' # 'hi' for Hindi
text_to_translate = 'how are you?'
# --- Step 2: Translate the text ---
try:
    # Translate the text. 'auto' can be used for the source.
    translated_text = GoogleTranslator(source=source_language, target=destination_language).translate(text_to_translate)

    # --- Step 3: Print the results ---
    print("--- Translation Result ---")
    print(f"Original ({source_language}): {text_to_translate}")
    print(f"Translated ({destination_language}): {translated_text}")
    print("--------------------------")

except Exception as e:
    print(f"An error occurred: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.1 MB/s eta 0:00:00
--- Translation Result ---
Original (ta): how are you?
Translated (hi): आप कैसे हैं?
--------------------------


### **combined code*****

In [ ]:
# ImpoImportrt the GoogleTranslator class from the new library
!pip install deep_translator
# Import the GoogleTranslator class from the new library
from deep_translator import GoogleTranslator

input1=input('Enter a text :')
lang=input('Enter the destination language :')
# --- Step 1: Set your languages and text here ---
#source_language = 'english'      # 'en' for English
destination_language = lang # 'hi' for Hindi
text_to_translate = input1
# --- Step 2: Translate the text ---
try:
    # Translate the text. 'auto' can be used for the source.
    translated_text = GoogleTranslator(target=destination_language).translate(text_to_translate)

    # --- Step 3: Print the results ---
    print("--- Translation Result ---")
   # print(f"Original ({source_language}): {text_to_translate}")
    print(f"Translated ({destination_language}): {translated_text}")
    print("--------------------------")

except Exception as e:
    print(f"An error occurred: {e}")

Enter a text :this is me
Enter the destination language :czech
--- Translation Result ---
Translated (czech): to jsem já
--------------------------


In [ ]:
!pip install deep_translator

from deep_translator import GoogleTranslator

def validate_input(text):
    """Validate input to ensure non-empty text."""
    if not text.strip():
        raise ValueError("Input text cannot be empty.")
    return text

def translate_text(text, target_language):
    """Translate the provided text to the target language."""
    try:
        # Translate the text automatically detecting the source language
        translated_text = GoogleTranslator(source='auto', target=target_language).translate(text)
        return translated_text
    except Exception as e:
        raise Exception(f"An error occurred during translation: {e}")

def main():
    # Step 1: Get input from the user
    input_text = input('Enter a text: ')
    target_language = input('Enter the destination language (e.g., "hi" for Hindi, "fr" for French, "es" for Spanish, etc.): ')

    # Step 2: Validate and prepare input
    try:
        input_text = validate_input(input_text)

        # Step 3: Translate text
        translated_text = translate_text(input_text, target_language)

        # Step 4: Display the result
        print("\n--- Translation Result ---")
        print(f"Original: {input_text}")
        print(f"Translated ({target_language}): {translated_text}")
        print("--------------------------")

    except Exception as e:
        print(f"Error: {e}")

# Run the main function
main()


Enter a text: there is a bank at the river bank
Enter the destination language (e.g., "hi" for Hindi, "fr" for French, "es" for Spanish, etc.): ta

--- Translation Result ---
Original: there is a bank at the river bank
Translated (ta): ஆற்றங்கரையில் ஒரு கரை உள்ளது
--------------------------


In [ ]:
!pip install deep_translator

from deep_translator import GoogleTranslator
import re

def auto_detect_context(text):
    """Automatically detect context based on the content of the text."""
    # Basic keyword-based context detection (simplified)
    medical_keywords = ["virus", "disease", "patient", "heart attack", "doctor", "hospital"]
    legal_keywords = ["contract", "court", "law", "legal", "claim"]
    formal_keywords = ["sir", "madam", "sincerely", "respectfully", "honorable"]
    informal_keywords = ["hey", "what's up", "yo", "wanna", "cool"]
    literary_keywords = ["poem", "dream", "magic", "love", "heartfelt"]

    # Check if any keywords match (case insensitive)
    if any(word in text.lower() for word in medical_keywords):
        return "technical"  # Medical/technical text
    elif any(word in text.lower() for word in legal_keywords):
        return "legal"  # Legal text
    elif any(word in text.lower() for word in formal_keywords):
        return "formal"  # Formal language
    elif any(word in text.lower() for word in informal_keywords):
        return "informal"  # Informal/casual language
    elif any(word in text.lower() for word in literary_keywords):
        return "literary"  # Literary or poetic text
    else:
        return "general"  # Default to general if no specific match

def clean_text_based_on_detected_context(text, context):
    """Modify text based on detected context for better translation accuracy."""
    if context == "technical":
        text = text.replace("heart attack", "myocardial infarction")
        text = text.replace("virus", "pathogen")
    elif context == "legal":
        text = text.replace("contract", "legal agreement")
        text = text.replace("claim", "legal claim")
    elif context == "formal":
        text = text.replace("you", "sir/madam")
        text = text.replace("can", "are able to")
    elif context == "informal":
        text = text.replace("you", "yo")
        text = text.replace("can", "wanna")
    elif context == "literary":
        text = text.replace("love", "ecstasy")
        text = text.replace("dream", "vision")
    return text

def validate_input(text):
    """Validate input to ensure non-empty text."""
    if not text.strip():
        raise ValueError("Input text cannot be empty.")
    return text

def translate_text(text, target_language, context):
    """Translate the provided text to the target language with context handling."""
    try:
        # Pre-process the text based on the detected context
        processed_text = clean_text_based_on_detected_context(text, context)

        # Translate the text automatically detecting the source language
        translated_text = GoogleTranslator(source='auto', target=target_language).translate(processed_text)
        return translated_text
    except Exception as e:
        raise Exception(f"An error occurred during translation: {e}")

def main():
    # Step 1: Get input from the user
    input_text = input('Enter a text: ')
    target_language = input('Enter the destination language (e.g., "hi" for Hindi, "fr" for French, "es" for Spanish, etc.): ')

    # Step 2: Automatically detect the context based on the input text
    context = auto_detect_context(input_text)

    # Step 3: Validate and prepare input
    try:
        input_text = validate_input(input_text)

        # Step 4: Translate text with automatic context detection
        translated_text = translate_text(input_text, target_language, context)

        # Step 5: Display the result
        print("\n--- Translation Result ---")
        print(f"Original: {input_text}")
        print(f"Translated ({target_language}): {translated_text}")
        print("--------------------------")

    except Exception as e:
        print(f"Error: {e}")

# Run the main function
main()


Enter a text: there is bank at river bank
Enter the destination language (e.g., "hi" for Hindi, "fr" for French, "es" for Spanish, etc.): ta

--- Translation Result ---
Original: there is bank at river bank
Translated (ta): ஆற்றங்கரையில் கரை உள்ளது
--------------------------


In [ ]:
!pip install deep_translator spacy

import spacy
from deep_translator import GoogleTranslator

# Load the spaCy model
nlp = spacy.load("en_core_web_sm")

def disambiguate_bank(text):
    """Disambiguate the word 'bank' based on surrounding context."""
    doc = nlp(text)

    for token in doc:
        if token.text.lower() == "bank":
            # Check surrounding words for financial or riverbank context
            # If there are words like "money", "finance", "loan", etc., assume financial context
            if any(kw in text.lower() for kw in ["money", "finance", "loan", "account", "deposit"]):
                return "financial"
            # If there are words like "river", "shore", "water", assume riverbank context
            elif any(kw in text.lower() for kw in ["river", "shore", "water", "stream"]):
                return "riverbank"
            else:
                return "general"  # Default to general if no clear context
    return "general"

def clean_text_based_on_context(text, context):
    """Modify the text based on detected context for better translation."""
    if context == "financial":
        text = text.replace("bank", "financial institution")
    elif context == "riverbank":
        text = text.replace("bank", "riverbank")
    return text

def validate_input(text):
    """Ensure the input text is not empty."""
    if not text.strip():
        raise ValueError("Input text cannot be empty.")
    return text

def translate_text(text, target_language, context):
    """Translate the provided text with context handling."""
    try:
        # Pre-process the text based on detected context
        processed_text = clean_text_based_on_context(text, context)

        # Translate the text using Google Translate
        translated_text = GoogleTranslator(source='auto', target=target_language).translate(processed_text)
        return translated_text
    except Exception as e:
        raise Exception(f"An error occurred during translation: {e}")

def main():
    # Step 1: Get input from the user
    input_text = input('Enter a text: ')
    target_language = input('Enter the destination language (e.g., "hi" for Hindi, "fr" for French, "es" for Spanish, etc.): ')

    # Step 2: Detect context automatically
    context = disambiguate_bank(input_text)

    # Step 3: Validate and prepare input
    try:
        input_text = validate_input(input_text)

        # Step 4: Translate text with context handling
        translated_text = translate_text(input_text, target_language, context)

        # Step 5: Display the result
        print("\n--- Translation Result ---")
        print(f"Original: {input_text}")
        print(f"Translated ({target_language}): {translated_text}")
        print("--------------------------")

    except Exception as e:
        print(f"Error: {e}")

# Run the main function
main()


Enter a text: there is a bank at riverbank
Enter the destination language (e.g., "hi" for Hindi, "fr" for French, "es" for Spanish, etc.): ta

--- Translation Result ---
Original: there is a bank at riverbank
Translated (ta): ஆற்றங்கரையில் ஒரு ஆற்றங்கரை உள்ளது
--------------------------


In [ ]:
!git clone https://github.com/vTuanpham/Large_dataset_translator.git

%cd Large_dataset_translator

!bash install.sh

Cloning into 'Large_dataset_translator'...
remote: Enumerating objects: 800, done.
remote: Counting objects: 100% (218/218), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 800 (delta 131), reused 133 (delta 76), pack-reused 582 (from 2)
Receiving objects: 100% (800/800), 149.60 MiB | 17.13 MiB/s, done.
Resolving deltas: 100% (436/436), done.
Updating files: 100% (75/75), done.
/content/Large_dataset_translator
Installing dependencies...
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of translators to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 7.7 MB

In [ ]:
# Step 1: Install necessary libraries
!pip install transformers datasets torch

# Step 2: Import required libraries
from transformers import MarianMTModel, MarianTokenizer
from datasets import load_dataset
from transformers import Trainer, TrainingArguments

# Step 3: Load the dataset
# For this example, we use the WMT dataset (English-German). You can replace this with your own parallel corpus
dataset = load_dataset('wmt14', 'de-en')  # You can replace 'de-en' with your language pair, e.g., 'en-fr', 'en-es'

# Step 4: Initialize the MarianMT tokenizer and model for translation
model_name = "Helsinki-NLP/opus-mt-en-de"  # Pre-trained model for English to German translation
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# Step 5: Preprocess and tokenize the dataset
# Tokenize the source (English) and target (German) texts
def tokenize_function(examples):
    return tokenizer(examples['translation']['en'], truncation=True, padding='max_length')

# Apply tokenization to the entire dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Step 6: Set up the training arguments
training_args = TrainingArguments(
    output_dir='./results',          # output directory to save model and logs
    evaluation_strategy="epoch",     # Evaluation after each epoch
    learning_rate=2e-5,              # Learning rate
    per_device_train_batch_size=16,  # Batch size for training
    per_device_eval_batch_size=64,   # Batch size for evaluation
    num_train_epochs=3,              # Number of epochs to train
    weight_decay=0.01,               # Weight decay for optimization
)

# Step 7: Initialize the Trainer
trainer = Trainer(
    model=model,                        # The model to train
    args=training_args,                 # Training arguments
    train_dataset=tokenized_datasets['train'],  # Training dataset
    eval_dataset=tokenized_datasets['test'],    # Evaluation dataset
)

# Step 8: Train the model
trainer.train()

# Step 9: Save the fine-tuned model
model.save_pretrained('./fine_tuned_model')
tokenizer.save_pretrained('./fine_tuned_model')

# Step 10: Use the trained model for inference
def translate_text(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", padding=True)
    outputs = model.generate(inputs['input_ids'])
    translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translated_text

# Step 11: Test the model
test_sentences = [
    "This is a test sentence.",
    "How are you today?",
    "I love machine learning."
]

for sentence in test_sentences:
    translated = translate_text(sentence, model, tokenizer)
    print(f"Original: {sentence}")
    print(f"Translated: {translated}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/4508785 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3003 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/298M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Map:   0%|          | 0/4508785 [00:00<?, ? examples/s]

TypeError: list indices must be integers or slices, not str

In [ ]:
# Step 1: Install necessary libraries
!pip install transformers datasets torch

# Step 2: Import required libraries
from transformers import MarianMTModel, MarianTokenizer
from datasets import load_dataset
from transformers import Trainer, TrainingArguments

# Step 3: Load the dataset
# For this example, we use the WMT dataset (English-German). You can replace this with your own parallel corpus
dataset = load_dataset('wmt14', 'de-en')  # You can replace 'de-en' with your language pair, e.g., 'en-fr', 'en-es'

# Step 4: Initialize the MarianMT tokenizer and model for translation
model_name = "Helsinki-NLP/opus-mt-en-de"  # Pre-trained model for English to German translation
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# Step 5: Preprocess and tokenize the dataset
# Modify the tokenize function to handle the dataset structure properly
def tokenize_function(examples):
    # Assuming 'en' is the source and 'de' is the target
    return tokenizer(examples['en'], truncation=True, padding='max_length')

# Apply tokenization to the entire dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Step 6: Set up the training arguments
training_args = TrainingArguments(
    output_dir='./results',          # output directory to save model and logs
    evaluation_strategy="epoch",     # Evaluation after each epoch
    learning_rate=2e-5,              # Learning rate
    per_device_train_batch_size=16,  # Batch size for training
    per_device_eval_batch_size=64,   # Batch size for evaluation
    num_train_epochs=3,              # Number of epochs to train
    weight_decay=0.01,               # Weight decay for optimization
)

# Step 7: Initialize the Trainer
trainer = Trainer(
    model=model,                        # The model to train
    args=training_args,                 # Training arguments
    train_dataset=tokenized_datasets['train'],  # Training dataset
    eval_dataset=tokenized_datasets['test'],    # Evaluation dataset
)

# Step 8: Train the model
trainer.train()

# Step 9: Save the fine-tuned model
model.save_pretrained('./fine_tuned_model')
tokenizer.save_pretrained('./fine_tuned_model')

# Step 10: Use the trained model for inference
def translate_text(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", padding=True)
    outputs = model.generate(inputs['input_ids'])
    translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translated_text

# Step 11: Test the model
test_sentences = [
    "This is a test sentence.",
    "How are you today?",
    "I love machine learning."
]

for sentence in test_sentences:
    translated = translate_text(sentence, model, tokenizer)
    print(f"Original: {sentence}")
    print(f"Translated: {translated}")


Map:   0%|          | 0/4508785 [00:00<?, ? examples/s]

KeyError: 'en'

In [ ]:
# Step 1: Install necessary libraries
!pip install transformers datasets torch pandas

# Step 2: Import required libraries
from transformers import MarianMTModel, MarianTokenizer
from datasets import Dataset
from transformers import Trainer, TrainingArguments
import pandas as pd

# Step 3: Load your custom dataset (replace with the path to your dataset)
df = pd.read_csv("path/to/your/custom_dataset.csv")  # Modify path
dataset = Dataset.from_pandas(df)  # Convert to HuggingFace dataset format

# Step 4: Initialize the MarianMT tokenizer and model for translation
model_name = "Helsinki-NLP/opus-mt-en-de"  # Example: English to German model
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# Step 5: Preprocess and tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples['source'], truncation=True, padding='max_length')

# Apply tokenization to the entire dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Step 6: Set up the training arguments
training_args = TrainingArguments(
    output_dir='./results',          # output directory to save model and logs
    evaluation_strategy="epoch",     # Evaluation after each epoch
    learning_rate=2e-5,              # Learning rate
    per_device_train_batch_size=16,  # Batch size for training
    per_device_eval_batch_size=64,   # Batch size for evaluation
    num_train_epochs=3,              # Number of epochs to train
    weight_decay=0.01,               # Weight decay for optimization
)

# Step 7: Initialize the Trainer
trainer = Trainer(
    model=model,                        # The model to train
    args=training_args,                 # Training arguments
    train_dataset=tokenized_datasets,   # Training dataset
    eval_dataset=tokenized_datasets,    # Evaluation dataset (you can modify this)
)

# Step 8: Train the model
trainer.train()

# Step 9: Save the fine-tuned model
model.save_pretrained('./fine_tuned_model')
tokenizer.save_pretrained('./fine_tuned_model')

# Step 10: Use the trained model for inference
def translate_text(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", padding=True)
    outputs = model.generate(inputs['input_ids'])
    translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translated_text

# Step 11: Test the model
test_sentences = [
    "This is a test sentence.",
    "How are you today?",
    "I love machine learning."
]

for sentence in test_sentences:
    translated = translate_text(sentence, model, tokenizer)
    print(f"Original: {sentence}")
    print(f"Translated: {translated}")


In [ ]:
# Install necessary libraries
!pip install fuzzywuzzy translators datasets tqdm tenacity memoization

import translators as ts
from datasets import load_dataset
from tqdm import tqdm
from fuzzywuzzy import fuzz
from tenacity import retry, stop_after_attempt, wait_fixed
from memoization import cached

# Load dataset (e.g., WMT)
dataset = load_dataset('wmt14', 'de-en')

# Translation function with retry logic and caching
@retry(stop=stop_after_attempt(3), wait=wait_fixed(2))
@cached()
def translate_text_with_retry_and_cache(text, dest_lang="es"):
    # Use the correct function call for Google Translate from the translators library
    return ts.translate_text(text, translator='google', to_language=dest_lang)

# Example of using the dataset and translating
translated_sentences = []
for example in tqdm(dataset['train'], desc="Translating"):
    # Access the English text correctly from the nested structure
    sentence = example['translation']['en']
    translated = translate_text_with_retry_and_cache(sentence, dest_lang="es")
    translated_sentences.append(translated)

# Check similarity of two translated sentences
similarity = fuzz.ratio(translated_sentences[0], translated_sentences[1])
print(f"Similarity between first two translations: {similarity}%")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of pathos to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of pathos to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 71.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 85.3 MB/s eta 0:00:00
  Installing build dependencies ... done


new tamil

In [ ]:
# Uninstall current transformers to avoid conflicts
!pip uninstall transformers -y -q

# Install a specific older version of transformers that includes GenerationMixin
!pip install -q transformers==4.30.0 datasets sentencepiece sacrebleu accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 kB 17.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 93.2 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


In [ ]:
# Clone the repo
!rm -rf Tamil-English-Dataset
!git clone https://github.com/Ishikahooda/Tamil-English-Dataset.git

# Show dataset folder
!ls -la Tamil-English-Dataset/Dataset || true


Cloning into 'Tamil-English-Dataset'...
remote: Enumerating objects: 39, done.
remote: Total 39 (delta 0), reused 0 (delta 0), pack-reused 39 (from 1)
Receiving objects: 100% (39/39), 41.09 MiB | 36.78 MiB/s, done.
Resolving deltas: 100% (11/11), done.
total 149564
drwxr-xr-x 2 root root     4096 Oct 21 13:05 .
drwxr-xr-x 4 root root     4096 Oct 21 13:05 ..
-rw-r--r-- 1 root root  6109990 Oct 21 13:05 data.en1
-rw-r--r-- 1 root root  6134861 Oct 21 13:05 data.en2
-rw-r--r-- 1 root root  6131501 Oct 21 13:05 data.en3
-rw-r--r-- 1 root root  6363259 Oct 21 13:05 data.en4
-rw-r--r-- 1 root root  7019676 Oct 21 13:05 data.en5
-rw-r--r-- 1 root root  5549365 Oct 21 13:05 data.en6
-rw-r--r-- 1 root root 18840648 Oct 21 13:05 data.ta1
-rw-r--r-- 1 root root 18892439 Oct 21 13:05 data.ta2
-rw-r--r-- 1 root root 18906253 Oct 21 13:05 data.ta3
-rw-r--r-- 1 root root 19731976 Oct 21 13:05 data.ta4
-rw-r--r-- 1 root root 22032333 Oct 21 13:05 data.ta5
-rw-r--r-- 1 root root 17417842 Oct 21 13:05 

In [ ]:
import os

DATA_DIR = "Tamil-English-Dataset/Dataset"
print("Files in Dataset folder:\n", os.listdir(DATA_DIR))

# Let's look at first 10 lines of each file
for fn in os.listdir(DATA_DIR):
    path = os.path.join(DATA_DIR, fn)
    if os.path.isfile(path):
        print(f"\n--- {fn} ---")
        !head -n 10 "$path"


Files in Dataset folder:
 ['data.en3', 'data.en2', 'data.ta5', 'data.en4', 'data.ta2', 'data.ta4', 'data.ta6', 'data.ta1', 'data.en1', 'data.ta3', 'data.en5', 'data.en6']

--- data.en3 ---
we said firmly no , ' he said .
and supper being ended , the devil having now put into the heart of judas iscariot , simon 's son , to betray him .
a small explosion destroyed a bicycle on a street crowded with shoppers preparing for friday , the muslim holy day .
he said to them , have you received the holy ghost since you believed ? and they said to him , we have not so much as heard whether there be any holy ghost .
they have no income and are subjected to stringent travel restrictions and police reporting requirements .
the london evening standard headline stated , ' â £ 9 million for 700 reserve strikebusting firefighters ' .
his speech was interrupted on several occasions by applause and cries of jubilation .
captain jaye burnett .
now there 's only one way you 're gonna find peace .
i 'll fini

In [ ]:
import pandas as pd
import os

DATA_DIR = "Tamil-English-Dataset/Dataset"

# Tamil and English files sorted by number
tam_files = sorted([f for f in os.listdir(DATA_DIR) if f.startswith("data.ta")])
eng_files = sorted([f for f in os.listdir(DATA_DIR) if f.startswith("data.en")])

all_pairs = []

for t_file, e_file in zip(tam_files, eng_files):
    with open(os.path.join(DATA_DIR, t_file), encoding='utf-8') as ft, \
         open(os.path.join(DATA_DIR, e_file), encoding='utf-8') as fe:
        tam_lines = [l.strip() for l in ft if l.strip()]
        eng_lines = [l.strip() for l in fe if l.strip()]
        n = min(len(tam_lines), len(eng_lines))
        all_pairs.extend(zip(tam_lines[:n], eng_lines[:n]))

# Create DataFrame
df = pd.DataFrame(all_pairs, columns=['tamil', 'english'])
print("Total parallel pairs loaded:", len(df))
df.head()


Total parallel pairs loaded: 289451


,tamil,english
0,ராஜாவாகிய ஆகாஸ் அரசாளும்போது தம்முடைய பாதகத்தி...,"moreover all the vessels , which king ahaz in ..."
1,சர்வதேச நாணய நிதியம் இலங்கைக்கு கடன் வழங்கினால...,similar conditions will be imposed if the sri ...
2,தற்போது அதற்கு எதிராக வாதாடுகிறார் சர்வதேச சட...,now kornelius argues the opposite instead of e...
3,அமெரிக்காவின் மூன்றாம் பெரிய கார் தயாரிப்பு நி...,chrysler the third largest us automaker filed ...
4,மேலும் இனைவிட்டு தலிபானால் வெளியேற்றப்பட்ட 199...,"moreover , khan has been in exile in iran for ..."


In [ ]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(df, test_size=0.10, random_state=42)
valid, test = train_test_split(temp, test_size=0.5, random_state=42)
print("train / valid / test sizes:", len(train), len(valid), len(test))


train / valid / test sizes: 260505 14473 14473


In [ ]:
from datasets import Dataset
from transformers import T5TokenizerFast

model_name = "t5-small"
tokenizer = T5TokenizerFast.from_pretrained(model_name)
source_prefix = "translate Tamil to English: "

def preprocess(df):
    inputs = [source_prefix + s for s in df['tamil'].tolist()]
    targets = df['english'].tolist()
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return Dataset.from_dict(model_inputs)

train_ds = preprocess(train)
valid_ds = preprocess(valid)
test_ds  = preprocess(test)

print(train_ds[0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
# -------------------------------
# 0️⃣ Install / Upgrade Libraries
# -------------------------------
!pip install --upgrade transformers datasets sentencepiece --quiet

# -------------------------------
# 1️⃣ Imports
# -------------------------------
import os
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import T5TokenizerFast, T5ForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments

# -------------------------------
# 2️⃣ Load Tamil-English Dataset
# -------------------------------
DATA_DIR = "Tamil-English-Dataset/Dataset"

# Tamil and English files sorted
tam_files = sorted([f for f in os.listdir(DATA_DIR) if f.startswith("data.ta")])
eng_files = sorted([f for f in os.listdir(DATA_DIR) if f.startswith("data.en")])

all_pairs = []

for t_file, e_file in zip(tam_files, eng_files):
    with open(os.path.join(DATA_DIR, t_file), encoding='utf-8') as ft, \
         open(os.path.join(DATA_DIR, e_file), encoding='utf-8') as fe:
        tam_lines = [l.strip() for l in ft if l.strip()]
        eng_lines = [l.strip() for l in fe if l.strip()]
        n = min(len(tam_lines), len(eng_lines))
        all_pairs.extend(zip(tam_lines[:n], eng_lines[:n]))

df = pd.DataFrame(all_pairs, columns=['tamil', 'english'])
print("Total sentence pairs loaded:", len(df))
df.head()

# -------------------------------
# 3️⃣ Split Dataset
# -------------------------------
train, temp = train_test_split(df, test_size=0.10, random_state=42)
valid, test = train_test_split(temp, test_size=0.5, random_state=42)
print("Train / Valid / Test sizes:", len(train), len(valid), len(test))

# -------------------------------
# 4️⃣ Tokenize Dataset
# -------------------------------
model_name = "t5-small"
tokenizer = T5TokenizerFast.from_pretrained(model_name)
source_prefix = "translate Tamil to English: "

def preprocess(df):
    inputs = [source_prefix + s for s in df['tamil'].tolist()]
    targets = df['english'].tolist()
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return Dataset.from_dict(model_inputs)

train_ds = preprocess(train)
valid_ds = preprocess(valid)
test_ds  = preprocess(test)

# -------------------------------
# 5️⃣ Load Model
# -------------------------------
model = T5ForConditionalGeneration.from_pretrained("t5-small")

# -------------------------------
# 6️⃣ Training Arguments
# -------------------------------
training_args = Seq2SeqTrainingArguments(
    output_dir="./tamil2english_model",
    evaluation_strategy="steps",
    eval_steps=1000,
    save_steps=1000,
    save_total_limit=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    weight_decay=0.01,
    num_train_epochs=1,
    predict_with_generate=True,
    fp16=True,
    logging_steps=500,
)

# -------------------------------
# 7️⃣ Trainer
# -------------------------------
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    tokenizer=tokenizer,
)

# -------------------------------
# 8️⃣ Train Model
# -------------------------------
trainer.train()

# -------------------------------
# 9️⃣ Translate Function
# -------------------------------
def translate_tamil(sentence):
    inputs = tokenizer("translate Tamil to English: " + sentence, return_tensors="pt", truncation=True, max_length=128)
    outputs = model.generate(**inputs, max_length=128, num_beams=4)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# -------------------------------
# 🔟 Test Translator
# -------------------------------
example = "நான் பள்ளிக்கு போகிறேன்"
print("Tamil:", example)
print("English:", translate_tamil(example))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.3/506.3 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.9/193.9 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.6/221.6 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.3/377.3 kB 35.5 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/torch_xla/experimental/gru.py:113: SyntaxWarning: invalid escape sequence '\_'
  * **h_n**: tensor of shape :math:`(D * \text{num\_layers}, H_{out})` or
/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:82: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Total sentence pairs loaded: 289451
Train / Valid / Test sizes: 260505 14473 14473


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
# -------------------------------
# 0️⃣ Install libraries
# -------------------------------
!pip install -q transformers datasets sentencepiece sacrebleu --upgrade

# -------------------------------
# 1️⃣ Imports
# -------------------------------
import torch
from datasets import load_dataset
from transformers import T5TokenizerFast, T5ForConditionalGeneration
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

# -------------------------------
# 2️⃣ Load Tatoeba Tamil-English Dataset
# -------------------------------
dataset = load_dataset("tatoeba", "ta")

# Keep only English translation
def filter_eng(batch):
    # filter out rows where English translation is missing
    return batch["translation"].get("en") is not None

dataset = dataset["train"].filter(filter_eng)

print("Number of Tamil-English sentences:", len(dataset))
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
val_dataset = dataset["test"]

# -------------------------------
# 3️⃣ Load Model & Tokenizer
# -------------------------------
model_name = "t5-small"
tokenizer = T5TokenizerFast.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# -------------------------------
# 4️⃣ Preprocess / Tokenize
# -------------------------------
max_input_length = 128
max_target_length = 128
source_prefix = "translate Tamil to English: "

def preprocess(batch):
    inputs = [source_prefix + s["ta"] for s in batch["translation"]]
    targets = [s["en"] for s in batch["translation"]]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=max_target_length, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train_dataset.map(preprocess, batched=True)
val_dataset = val_dataset.map(preprocess, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# -------------------------------
# 5️⃣ Data Collator
# -------------------------------
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# -------------------------------
# 6️⃣ Training Arguments
# -------------------------------
training_args = Seq2SeqTrainingArguments(
    output_dir="./tamil2english_model",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    predict_with_generate=True,
    evaluation_strategy="steps",
    save_strategy="steps",
    save_steps=500,
    eval_steps=500,
    logging_steps=100,
    learning_rate=5e-5,
    num_train_epochs=1,
    weight_decay=0.01,
    save_total_limit=2,
    fp16=True,
    report_to="none"
)

# -------------------------------
# 7️⃣ Trainer
# -------------------------------
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

# -------------------------------
# 8️⃣ Train Model
# -------------------------------
trainer.train()

# -------------------------------
# 9️⃣ Translation Function
# -------------------------------
def translate_tamil(sentence):
    inputs = tokenizer(source_prefix + sentence, return_tensors="pt", truncation=True, max_length=128)
    outputs = model.generate(**inputs, max_length=128, num_beams=4)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# -------------------------------
# 🔟 Test Translator
# -------------------------------
example = "நான் பள்ளிக்கு போகிறேன்"
print("Tamil:", example)
print("English:", translate_tamil(example))


README.md: 0.00B [00:00, ?B/s]

tatoeba.py: 0.00B [00:00, ?B/s]

RuntimeError: Dataset scripts are no longer supported, but found tatoeba.py

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

# Load the English-to-Tamil model
model_name = "Helsinki-NLP/opus-mt-en-ta"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# Example Tamil sentence
sentence = "நான் பள்ளிக்கு போகிறேன்"

# Tokenize the input
inputs = tokenizer("translate English to Tamil: " + sentence, return_tensors="pt", padding=True)

# Generate translation
outputs = model.generate(**inputs, max_length=40)

# Decode the generated tokens
translated_sentence = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Original: {sentence}")
print(f"Translated: {translated_sentence}")


ModuleNotFoundError: Could not import module 'MarianMTModel'. Are this object's requirements defined correctly?

final code

In [ ]:
from transformers import pipeline

# Initialize the NLLB translator
translator = pipeline("translation", model="facebook/nllb-200-distilled-600M")

# Function to translate
def translate(text, src_lang, tgt_lang):
    result = translator(text, src_lang=src_lang, tgt_lang=tgt_lang)
    return result[0]['translation_text']

# Supported languages (example subset)
languages = {
    "English": "eng_Latn",
    "Tamil": "tam_Taml",
    "Hindi": "hin_Deva",
    "French": "fra_Latn",
    "Spanish": "spa_Latn"
}

# User input
print("Available languages:", ", ".join(languages.keys()))
src_lang_name = input("Enter source language: ")
tgt_lang_name = input("Enter target language: ")
text = input("Enter text to translate: ")

# Map names to NLLB language codes
src_lang = languages.get(src_lang_name)
tgt_lang = languages.get(tgt_lang_name)

if src_lang and tgt_lang:
    translated_text = translate(text, src_lang, tgt_lang)
    print(f"\nOriginal ({src_lang_name}): {text}")
    print(f"Translated ({tgt_lang_name}): {translated_text}")
else:
    print("Invalid language selection. Please check the language names.")


Device set to use cpu


Available languages: English, Tamil, Hindi, French, Spanish
Enter source language: Tamil
Enter target language: French
Enter text to translate:  என் பெயர் தௌஃபீக்.

Original (Tamil):  என் பெயர் தௌஃபீக்.
Translated (French): Je m'appelle Taufik.


**new** **code**

In [ ]:
from transformers import pipeline
import time # Import time to measure loading time

# --- Model Initialization ---
# Initialize the NLLB translator. This step downloads the model weights and can take a while.
print("Initializing NLLB-200 translator...")
start_time = time.time()
try:
    translator = pipeline("translation", model="facebook/nllb-200-distilled-600M")
    print(f"Translator loaded in {time.time() - start_time:.2f} seconds.")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please ensure you have 'transformers' and 'torch' installed.")
    exit()

# --- Translation Function ---
def translate(text, src_lang, tgt_lang):
    """Translates text using the NLLB pipeline."""
    # The NLLB model expects a list of texts
    result = translator(text, src_lang=src_lang, tgt_lang=tgt_lang)
    return result[0]['translation_text']

# --- Supported Languages (8 total: 3 specified + 5 low-resource) ---
# NLLB-200 Language codes (BCP-47 format)
languages = {
    # Specified Languages
    "English": "eng_Latn",
    "Tamil": "tam_Taml",
    "Hindi": "hin_Deva",

    # Five Low-Resource Languages
    # (Note: "Low-resource" status can be relative, but these are generally less resourced than French/Spanish)
    "Marathi": "mar_Deva", # India (Indo-Aryan, Devanagari script)
    "Nepali": "nep_Deva",   # Nepal/India (Indo-Aryan, Devanagari script)
    "Sinhala": "sin_Sinh",  # Sri Lanka (Indo-Aryan, Sinhala script)
    "Telugu": "tel_Telu",  # India (Dravidian, Telugu script)
    "Lao": "lao_Laoo"     # Laos (Tai-Kadai, Lao script)
}

# --- User Input and Execution ---
print("\n--- NLLB Translation Script ---")
print(f"Available languages ({len(languages)}): {', '.join(languages.keys())}")

# Collect user input
src_lang_name = input("Enter source language: ").strip()
tgt_lang_name = input("Enter target language: ").strip()
text = input("Enter text to translate: ").strip()

# Map names to NLLB language codes (case-insensitive search for better UX)
src_lang = languages.get(src_lang_name.title())
tgt_lang = languages.get(tgt_lang_name.title())

if src_lang and tgt_lang:
    print("\nTranslating...")
    try:
        translated_text = translate(text, src_lang, tgt_lang)
        print("--------------------------------------------------")
        print(f"Original ({src_lang_name.title()}): {text}")
        print(f"Translated ({tgt_lang_name.title()}): {translated_text}")
        print("--------------------------------------------------")
    except Exception as e:
        print(f"An error occurred during translation: {e}")
else:
    print("\nInvalid language selection. Please check the language names and try again.")
    print("Ensure the language name is capitalized correctly (e.g., 'English', not 'english').")

Initializing NLLB-200 translator...


Device set to use cpu


Translator loaded in 12.97 seconds.

--- NLLB Translation Script ---
Available languages (8): English, Tamil, Hindi, Marathi, Nepali, Sinhala, Telugu, Lao
Enter source language: English
Enter target language: Tamil
Enter text to translate: from transformers import pipeline import time # Import time to measure loading time  # --- Model Initialization --- # Initialize the NLLB translator. This step downloads the model weights and can take a while. print("Initializing NLLB-200 translator...") start_time = time.time() try:     translator = pipeline("translation", model="facebook/nllb-200-distilled-600M")     print(f"Translator loaded in {time.time() - start_time:.2f} seconds.") except Exception as e:     print(f"Error loading model: {e}")     print("Please ensure you have 'transformers' and 'torch' installed.")     exit()  # --- Translation Function --- def translate(text, src_lang, tgt_lang):     """Translates text using the NLLB pipeline."""     # The NLLB model expects a list of texts  

Your input_length: 844 is bigger than 0.9 * max_length: 200. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)



Translating...


KeyboardInterrupt: 

In [ ]:
from transformers import pipeline

# Initialize the NLLB translator
translator = pipeline("translation", model="facebook/nllb-200-distilled-600M")

# Function to translate
def translate(text, src_lang, tgt_lang):
    result = translator(text, src_lang=src_lang, tgt_lang=tgt_lang)
    return result[0]['translation_text']

# Supported languages (updated to include low-resource examples)
languages = {
    "English": "eng_Latn",
    "Tamil":   "tam_Taml",
    "Hindi":   "hin_Deva",
    "Yoruba":  "yor_Latn",
    "Igbo":    "ibo_Latn",
    "Nepali":  "npi_Deva",
    "Sinhala": "sin_Sinh",
    "Telugu":  "tel_Telu"
}

# User input
print("Available languages:", ", ".join(languages.keys()))
src_lang_name = input("Enter source language: ")
tgt_lang_name = input("Enter target language: ")
text = input("Enter text to translate: ")

# Map names to NLLB language codes
src_lang = languages.get(src_lang_name)
tgt_lang = languages.get(tgt_lang_name)

if src_lang and tgt_lang:
    translated_text = translate(text, src_lang, tgt_lang)
    print(f"\nOriginal ({src_lang_name}): {text}")
    print(f"Translated ({tgt_lang_name}): {translated_text}")
else:
    print("Invalid language selection. Please check the language names.")


Device set to use cpu


Available languages: English, Tamil, Hindi, Yoruba, Igbo, Nepali, Sinhala, Telugu
Enter source language: English
Enter target language: Tamil
Enter text to translate: there is a bank near the river bank

Original (English): there is a bank near the river bank
Translated (Tamil): ஆற்றின் கரையில் ஒரு கடற்கரை உள்ளது


In [ ]:
# Install ipywidgets
!pip install -q ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.5 MB/s eta 0:00:00


In [ ]:
from ipywidgets import Dropdown, Textarea, Button, VBox, HBox, Layout
from IPython.display import display
from transformers import pipeline
import time

# --- NLLB Model and Translation Logic ---

# NLLB-200 Language codes (8 total)
NLLB_LANGUAGES = {
    "English": "eng_Latn",
    "Tamil":   "tam_Taml",
    "Hindi":   "hin_Deva",
    "Yoruba":  "yor_Latn",
    "Igbo":    "ibo_Latn",
    "Nepali":  "nep_Deva",
    "Sinhala": "sin_Sinh",
    "Telugu":  "tel_Telu"
}
LANGUAGE_NAMES = list(NLLB_LANGUAGES.keys())

# Global variable for the translator pipeline
translator = None
MODEL_NAME = "facebook/nllb-200-distilled-600M"

def initialize_translator():
    """Initializes the NLLB translator pipeline."""
    global translator
    if translator is None:
        print(f"Loading {MODEL_NAME}...")
        start_time = time.time()
        try:
            # Load the model. This is the most time-consuming step.
            translator = pipeline("translation", model=MODEL_NAME)
            print(f"Translator loaded in {time.time() - start_time:.2f} seconds.")
        except Exception as e:
            print(f"Error loading model: {e}")
            print("Please ensure you have 'transformers' and 'torch' installed.")
            translator = None

def translate_text_nllb(text, src_lang_code, tgt_lang_code):
    """Translates text using the NLLB pipeline."""
    global translator
    if translator is None:
        return "ERROR: Translator model not loaded. Please run the initialization cell."

    try:
        # The pipeline function handles the translation
        result = translator(text, src_lang=src_lang_code, tgt_lang=tgt_lang_code)
        return result[0]['translation_text']
    except Exception as e:
        return f"Translation failed: {e}"

# --- ipywidgets GUI Implementation ---

# Language Dropdowns
src_lang_dropdown = Dropdown(
    options=LANGUAGE_NAMES,
    value='English',
    description='Source:',
    disabled=False,
    layout=Layout(width='45%')
)

tgt_lang_dropdown = Dropdown(
    options=LANGUAGE_NAMES,
    value='Tamil',
    description='Target:',
    disabled=False,
    layout=Layout(width='45%')
)

# Swap Button
swap_button = Button(description='🔁 Swap', layout=Layout(width='10%'))

def swap_languages_event(b):
    current_src = src_lang_dropdown.value
    current_tgt = tgt_lang_dropdown.value
    src_lang_dropdown.value = current_tgt
    tgt_lang_dropdown.value = current_src

swap_button.on_click(swap_languages_event)

# Input and Output Text Areas
input_textarea = Textarea(
    value='',
    placeholder='Enter text to translate',
    description='Input:',
    disabled=False,
    layout=Layout(width='80%', height='100px')
)

output_textarea = Textarea(
    value='',
    placeholder='Translated text will appear here',
    description='Output:',
    disabled=True, # Output should not be editable
    layout=Layout(width='80%', height='100px')
)

# Translate Button
translate_button = Button(
    description='TRANSLATE',
    disabled=False,
    button_style='primary', # Style the button
    tooltip='Click to translate',
    layout=Layout(width='auto')
)

def translate_button_event(b):
    input_text = input_textarea.value.strip()

    if not input_text:
        output_textarea.value = "Please enter text to translate."
        return

    src_lang_name = src_lang_dropdown.value
    tgt_lang_name = tgt_lang_dropdown.value

    src_code = NLLB_LANGUAGES.get(src_lang_name)
    tgt_code = NLLB_LANGUAGES.get(tgt_lang_name)

    if not src_code or not tgt_code:
        output_textarea.value = "Invalid language selection."
        return

    # Update UI while translating
    translate_button.description = "TRANSLATING..."
    translate_button.disabled = True

    # Perform the translation
    translated_text = translate_text_nllb(input_text, src_code, tgt_code)

    # Update output area
    output_textarea.disabled = False
    output_textarea.value = f"Original ({src_lang_name}): {input_text}\n\nTranslated ({tgt_lang_name}): {translated_text}"
    output_textarea.disabled = True

    # Restore button state
    translate_button.description = "TRANSLATE"
    translate_button.disabled = False


translate_button.on_click(translate_button_event)

# --- Layout and Display ---

# Arrange widgets
language_selection_box = HBox([src_lang_dropdown, swap_button, tgt_lang_dropdown])
input_box = VBox([input_textarea])
output_box = VBox([output_textarea])
button_box = HBox([translate_button], layout=Layout(justify_content='center')) # Center the button

# Main container
app_layout = VBox([
    language_selection_box,
    input_box,
    button_box,
    output_box
])

# Display the application
print("Displaying the translator interface...")
display(app_layout)

# Initialize the NLLB model (this will print "Loading...")
initialize_translator()

Displaying the translator interface...


Loading facebook/nllb-200-distilled-600M...


Device set to use cpu


Translator loaded in 11.00 seconds.


## **final output code with the user interface**

In [ ]:
from ipywidgets import Dropdown, Textarea, Button, VBox, HBox, Layout, HTML
from IPython.display import display
from transformers import pipeline
import time

# --- NLLB Model and Translation Logic ---

NLLB_LANGUAGES = {
    "English": "eng_Latn",
    "Tamil":   "tam_Taml",
    "Hindi":   "hin_Deva",
    "Yoruba":  "yor_Latn",
    "Igbo":    "ibo_Latn",
    "Nepali":  "nep_Deva",
    "Sinhala": "sin_Sinh",
    "Telugu":  "tel_Telu"
}
LANGUAGE_NAMES = list(NLLB_LANGUAGES.keys())

translator = None
MODEL_NAME = "facebook/nllb-200-distilled-600M"

def initialize_translator():
    """Initializes the NLLB translator pipeline."""
    global translator
    if translator is None:
        print(f"Loading {MODEL_NAME}...")
        start_time = time.time()
        try:
            translator = pipeline("translation", model=MODEL_NAME)
            print(f"✅ Translator loaded in {time.time() - start_time:.2f} seconds.")
        except Exception as e:
            print(f"❌ Error loading model: {e}")
            translator = None

def translate_text_nllb(text, src_lang_code, tgt_lang_code):
    """Translates text using the NLLB pipeline."""
    global translator
    if translator is None:
        return "ERROR: Translator model not loaded. Please run the initialization cell."
    try:
        result = translator(text, src_lang=src_lang_code, tgt_lang=tgt_lang_code)
        return result[0]['translation_text']
    except Exception as e:
        return f"Translation failed: {e}"

# --- ipywidgets GUI ---

# Titles
title_html = HTML(
    value="<h2 style='color:#2E86C1; text-align:center;'>🌍 NLLB Multi-Language Translator</h2>"
)

# Language Dropdowns
src_lang_dropdown = Dropdown(
    options=LANGUAGE_NAMES,
    value='English',
    description='Source:',
    layout=Layout(width='45%')
)

tgt_lang_dropdown = Dropdown(
    options=LANGUAGE_NAMES,
    value='Tamil',
    description='Target:',
    layout=Layout(width='45%')
)

swap_button = Button(description='🔁 Swap', layout=Layout(width='10%', height='35px'))

def swap_languages_event(b):
    src_lang_dropdown.value, tgt_lang_dropdown.value = tgt_lang_dropdown.value, src_lang_dropdown.value

swap_button.on_click(swap_languages_event)

# Input Label and Textarea
input_label = HTML("<b style='color:#1F618D;'>Input Text:</b>")
input_textarea = Textarea(
    placeholder='Enter text to translate...',
    layout=Layout(width='90%', height='100px')
)

# Output Label and Textarea
output_label = HTML("<b style='color:#117864;'>Translated Output:</b>")
output_textarea = Textarea(
    placeholder='Translated text will appear here...',
    disabled=True,
    layout=Layout(width='90%', height='100px')
)

# Translate Button
translate_button = Button(
    description='🚀 TRANSLATE',
    button_style='info',
    tooltip='Click to translate text',
    layout=Layout(width='200px', height='40px')
)

status_label = HTML("<i style='color:gray;'>Ready to translate...</i>")

def translate_button_event(b):
    input_text = input_textarea.value.strip()
    if not input_text:
        status_label.value = "<i style='color:red;'>⚠️ Please enter text to translate.</i>"
        return

    src_lang = src_lang_dropdown.value
    tgt_lang = tgt_lang_dropdown.value
    src_code = NLLB_LANGUAGES.get(src_lang)
    tgt_code = NLLB_LANGUAGES.get(tgt_lang)

    if not src_code or not tgt_code:
        status_label.value = "<i style='color:red;'>⚠️ Invalid language selection.</i>"
        return

    translate_button.description = "⏳ Translating..."
    translate_button.disabled = True
    status_label.value = f"<i style='color:gray;'>Translating from {src_lang} → {tgt_lang}...</i>"

    translated_text = translate_text_nllb(input_text, src_code, tgt_code)

    output_textarea.disabled = False
    output_textarea.value = translated_text
    output_textarea.disabled = True

    translate_button.description = "🚀 TRANSLATE"
    translate_button.disabled = False
    status_label.value = "<i style='color:green;'>✅ Translation complete!</i>"

translate_button.on_click(translate_button_event)

# --- Layout ---
language_box = HBox([src_lang_dropdown, swap_button, tgt_lang_dropdown],
                    layout=Layout(justify_content='space-between', margin='10px 0'))

button_box = HBox([translate_button],
                  layout=Layout(justify_content='center', margin='10px 0'))

app_layout = VBox([
    title_html,
    language_box,
    input_label, input_textarea,
    button_box,
    output_label, output_textarea,
    status_label
], layout=Layout(padding='15px', border='2px solid #D6DBDF', border_radius='10px', width='100%'))

print("Displaying the translator interface...")
display(app_layout)

# Initialize the model
initialize_translator()

Displaying the translator interface...


Loading facebook/nllb-200-distilled-600M...


Device set to use cuda:0


✅ Translator loaded in 10.18 seconds.


In [ ]:
from ipywidgets import Dropdown, Textarea, Button, VBox, HBox, Layout, HTML
from IPython.display import display
from transformers import pipeline
import time

# --- NLLB Model and Translation Logic ---

NLLB_LANGUAGES = {
    "English": "eng_Latn",
    "Tamil":   "tam_Taml",
    "Hindi":   "hin_Deva",
    "Yoruba":  "yor_Latn",
    "Igbo":    "ibo_Latn",
    "Nepali":  "nep_Deva",
    "Sinhala": "sin_Sinh",
    "Telugu":  "tel_Telu"
}
LANGUAGE_NAMES = list(NLLB_LANGUAGES.keys())

translator = None
MODEL_NAME = "facebook/nllb-200-distilled-600M"

def initialize_translator():
    """Initializes the NLLB translator pipeline."""
    global translator
    if translator is None:
        print(f"Loading {MODEL_NAME}...")
        start_time = time.time()
        try:
            translator = pipeline("translation", model=MODEL_NAME)
            print(f"✅ Translator loaded in {time.time() - start_time:.2f} seconds.")
        except Exception as e:
            print(f"❌ Error loading model: {e}")
            translator = None

def translate_text_nllb(text, src_lang_code, tgt_lang_code):
    """Translates text using the NLLB pipeline."""
    global translator
    if translator is None:
        return "ERROR: Translator model not loaded. Please run the initialization cell."
    try:
        result = translator(text, src_lang=src_lang_code, tgt_lang=tgt_lang_code)
        return result[0]['translation_text']
    except Exception as e:
        return f"Translation failed: {e}"

# --- ipywidgets GUI ---

# Titles
title_html = HTML(
    value="<h2 style='color:#2E86C1; text-align:center;'>🌍 NLLB Multi-Language Translator</h2>"
)

# Language Dropdowns
src_lang_dropdown = Dropdown(
    options=LANGUAGE_NAMES,
    value='English',
    description='Source:',
    layout=Layout(width='45%')
)

tgt_lang_dropdown = Dropdown(
    options=LANGUAGE_NAMES,
    value='Tamil',
    description='Target:',
    layout=Layout(width='45%')
)

swap_button = Button(description='🔁 Swap', layout=Layout(width='10%', height='35px'))

def swap_languages_event(b):
    src_lang_dropdown.value, tgt_lang_dropdown.value = tgt_lang_dropdown.value, src_lang_dropdown.value

swap_button.on_click(swap_languages_event)

# Input Label and Textarea
input_label = HTML("<b style='color:#1F618D;'>Input Text:</b>")
input_textarea = Textarea(
    placeholder='Enter text to translate...',
    layout=Layout(width='90%', height='100px')
)

# Output Label and Textarea
output_label = HTML("<b style='color:#117864;'>Translated Output:</b>")
output_textarea = Textarea(
    placeholder='Translated text will appear here...',
    disabled=True,
    layout=Layout(width='90%', height='100px')
)

# Translate Button
translate_button = Button(
    description='🚀 TRANSLATE',
    button_style='info',
    tooltip='Click to translate text',
    layout=Layout(width='200px', height='40px')
)

status_label = HTML("<i style='color:gray;'>Ready to translate...</i>")

def translate_button_event(b):
    input_text = input_textarea.value.strip()
    if not input_text:
        status_label.value = "<i style='color:red;'>⚠ Please enter text to translate.</i>"
        return

    src_lang = src_lang_dropdown.value
    tgt_lang = tgt_lang_dropdown.value
    src_code = NLLB_LANGUAGES.get(src_lang)
    tgt_code = NLLB_LANGUAGES.get(tgt_lang)

    if not src_code or not tgt_code:
        status_label.value = "<i style='color:red;'>⚠ Invalid language selection.</i>"
        return

    translate_button.description = "⏳ Translating..."
    translate_button.disabled = True
    status_label.value = f"<i style='color:gray;'>Translating from {src_lang} → {tgt_lang}...</i>"

    translated_text = translate_text_nllb(input_text, src_code, tgt_code)

    output_textarea.disabled = False
    output_textarea.value = translated_text
    output_textarea.disabled = True

    translate_button.description = "🚀 TRANSLATE"
    translate_button.disabled = False
    status_label.value = "<i style='color:green;'>✅ Translation complete!</i>"

translate_button.on_click(translate_button_event)

# --- Layout ---
language_box = HBox([src_lang_dropdown, swap_button, tgt_lang_dropdown],
                    layout=Layout(justify_content='space-between', margin='10px 0'))

button_box = HBox([translate_button],
                  layout=Layout(justify_content='center', margin='10px 0'))

app_layout = VBox([
    title_html,
    language_box,
    input_label, input_textarea,
    button_box,
    output_label, output_textarea,
    status_label
], layout=Layout(padding='15px', border='2px solid #D6DBDF', border_radius='10px', width='100%'))

print("Displaying the translator interface...")
display(app_layout)

# Initialize the model
initialize_translator()

KeyboardInterrupt: 